# Biochemistry: ESM-2 fine-tuning

In this notebook we will finetune an ESM model, specifically, **ESM-2 (8M parameters; 6 encoder blocks)** on the ProMelt dataset. We will compare four finetuning strategies all using the same regression-head architecture for validation metrics and computational cost.

1. **Frozen backbone:** train the regression head only.
2. **Freeze first half:** freeze ESM-2's embeddings and blocks 0–2; fine-tune blocks 3–5 and the head.
3. **Full fine-tuning:** train all ESM-2 parameters and the head.
4. **LoRA throughout:** freeze pretrained ESM-2 weights; train LoRA adapters on query/value projections in **all 6 blocks** plus the head.

All four run for the **same number of complete epochs**, using the same split, preprocessing, optimizer family, batch size and regression head.

*Note:* Different fine tuning approaches may have different optimal learning-rates. The notebook allows you to try varying the learning rate for each approach.

## Install dependencies and download ESM-2 weights
Run if you do not have packages installed or weights downlaoded. The model's weights and tokenizer will be cached in the Hugging Face cache.

In [ ]:
%pip install -q -U "transformers>=4.43,<5" "peft>=0.12,<1" "accelerate>=0.30" "huggingface_hub>=0.24" pandas numpy scikit-learn matplotlib psutil
%pip install torch torchvision torchaudio
%pip uninstall -y torchao

# The following downloads and caches both the pretrained weights and tokenizer.
# Requires internet on the first run (subsequent runs can use the cache).
from huggingface_hub import snapshot_download
MODEL_NAME = "facebook/esm2_t6_8M_UR50D"  # six encoder layers, ~8 million parameters
model_cache_dir = snapshot_download(repo_id=MODEL_NAME)
print("ESM-2 downloaded/cached:", model_cache_dir)


## Imports, set seeds and experiment configuration

Set `NUM_EPOCHS` to a common fixed value for all methods.

In [ ]:
import gc
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from transformers import AutoTokenizer, EsmModel
from peft import LoraConfig, get_peft_model

RANDOM_SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_EPOCHS = 15            # Exactly this many for EVERY approach; no early stopping
BATCH_SIZE = 32            # Same for every approach; reduce if full fine-tuning runs out of GPU memory
MAX_LENGTH = 700           # Protein residues, excluding ESM special tokens
WEIGHT_DECAY = 0.01
#LORA parameters
LORA_RANK = 8
LORA_ALPHA = 16
LORA_DROPOUT = 0.05
#you can try different learning rates for different fine tuning
#smaller learning rates are likely better for less frozen model, to avoid deviating too much from pretrained model
LEARNING_RATES = {
    "Frozen backbone": 1e-3,
    "Freeze first half": 2e-4,
    "Full fine-tuning": 2e-4,
    "LoRA all layers": 3e-4,
}

def set_seed(seed=RANDOM_SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed()
print("Device:", DEVICE)
print("Torch:", torch.__version__)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Max input length includes <cls> and <eos>, so sequences may have up to 800 residues.
# We are using the ESM-2 tokenizer here
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

##  Load ProMelt data and tokenize with ESM-2


In [ ]:
df = pd.read_csv("full_promelt_seq.csv")
df = df.copy()
df["sequence"] = df["sequence"].astype("string").str.strip().str.upper()
df["Tm"] = pd.to_numeric(df["Tm"], errors="coerce")
#remove rows with invalid amino acids
df = df[df["sequence"].str.count("X")==0]
df = df[df["sequence"].str.count("U")==0]
df = df[df["sequence"].str.count("Z")==0]
df = df[df["sequence"].str.count("B")==0]
valid_length = df["sequence"].str.len().between(1, MAX_LENGTH).fillna(False)
df = df.loc[valid_length & df["Tm"].notna()].reset_index(drop=True)

train_df, val_df = train_test_split(df, test_size=0.2, random_state=RANDOM_SEED)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print(f"Filtered total: {len(df):,}; train: {len(train_df):,}; validation: {len(val_df):,}")
print("Sequence length range:", df["sequence"].str.len().min(), "-", df["sequence"].str.len().max())

#makes a dataset with sequences and Tm
class ProteinTmDataset(Dataset):
    def __init__(self, frame):
        self.sequences = frame["sequence"].tolist()
        self.labels = frame["Tm"].to_numpy(dtype=np.float32)

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, index):
        return self.sequences[index], float(self.labels[index])

#puts together batch with padding
def collate_batch(items):
    sequences, temperatures = zip(*items)
    tokens = tokenizer(
        list(sequences), padding=True, truncation=False, return_tensors="pt"
    )
    return {
        "input_ids": tokens["input_ids"],
        "attention_mask": tokens["attention_mask"],
        "labels": torch.tensor(temperatures, dtype=torch.float32),
    }

train_dataset = ProteinTmDataset(train_df)
val_dataset = ProteinTmDataset(val_df)

def make_loaders(seed=RANDOM_SEED):
    # Fresh seeded generator for identical shuffle order across methods.
    gen = torch.Generator().manual_seed(seed)
    return (
        DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,
                   generator=gen, collate_fn=collate_batch),
        DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False,
                   collate_fn=collate_batch),
    )

train_loader, val_loader = make_loaders()
example = next(iter(train_loader))
print("ESM token IDs:", tuple(example["input_ids"].shape),
      "(includes beginning/end tokens and batch padding)")

## Model definition and the four fine-tuning configurations

Every experiment reinitializes the same pretrained ESM-2 checkpoint and identically seeded regression head. The model mean-pools only real amino acid tokens, excluding ESM's start/end and padding tokens. All configurations use the same head, validation data, and MSE training loss.

ESM-2 8M has 6 blocks, so the first-half strategy freezes the embedding and the first 3 blocks. LoRA is added to both query and value projections in each of the 6 blocks; the original backbone remains frozen.

In [ ]:
# this is a class for our regression head
class EsmTmRegressor(nn.Module):
    def __init__(self, backbone, hidden_size, dropout=0.2):
        super().__init__()
        self.esm = backbone
        self.regressor = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 1),
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.esm(input_ids=input_ids, attention_mask=attention_mask)
        embeddings = outputs.last_hidden_state
        # Exclude PAD, CLS and EOS tokens from mean-pooling.
        residue_mask = (
            attention_mask.bool()
            & (input_ids != tokenizer.cls_token_id)
            & (input_ids != tokenizer.eos_token_id)
            & (input_ids != tokenizer.pad_token_id)
        )
        weights = residue_mask.unsqueeze(-1).to(embeddings.dtype)
        pooled = (embeddings * weights).sum(dim=1) / weights.sum(dim=1).clamp(min=1)
        return self.regressor(pooled).squeeze(-1)

METHODS = ["Frozen backbone", "Freeze first half", "Full fine-tuning", "LoRA all layers"]

def build_model(method):
    if method not in METHODS:
        raise ValueError(f"Unknown method: {method}")
    # Identical head initialization and dropout randomness across methods.
    set_seed(RANDOM_SEED)
    #load ESM model
    backbone = EsmModel.from_pretrained(MODEL_NAME, add_pooling_layer=False)
    #remove gradients from all parameters
    for param in backbone.parameters():
        param.requires_grad = False
    #get the encoder portion of ESM
    blocks = backbone.encoder.layer
    
    #add gradients back for layers that should be unfrozen for each approach
    if method == "Freeze first half":
        for block in blocks[len(blocks)//2:]:
            for param in block.parameters():
                param.requires_grad = True
    elif method == "Full fine-tuning":
        for param in backbone.parameters():
            param.requires_grad = True
    elif method == "LoRA all layers":
        config = LoraConfig(
            r=LORA_RANK, lora_alpha=LORA_ALPHA,
            lora_dropout=LORA_DROPOUT,
            bias="none", target_modules=["query", "value"],
        )
        backbone = get_peft_model(backbone, config)
    #add regression head to model
    model = EsmTmRegressor(backbone, backbone.config.hidden_size)
    return model.to(DEVICE)

#compare total and traininable parameters across approachs
def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

# Verify all four configurations without occupying the GPU with four models.
model_summaries = []
for method in METHODS:
    model = build_model(method)
    total, trainable = count_parameters(model)
    model_summaries.append({
        "Method": method, "Total parameters": total,
        "Trainable parameters": trainable,
        "Trainable %": 100 * trainable / total,
        "Learning rate": LEARNING_RATES[method],
    })
    #check that LoRA was set up correctly
    if method == "LoRA all layers":
        lora_block_ids = {
            int(name.split(".layer.")[1].split(".")[0])
            for name, param in model.named_parameters()
            if "lora_" in name and param.requires_grad and ".layer." in name
        }
        assert lora_block_ids == set(range(6)), f"LoRA missing blocks: {lora_block_ids}"
    del model
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

parameter_comparison = pd.DataFrame(model_summaries)
display(parameter_comparison)

## Training and validation functions, time and memory measurement

For each epoch, measure train time, validation time, and peak memory. On CUDA, timing uses `torch.cuda.synchronize()` so GPU operations complete before the clock is read. CUDA allocated and reserved memory peaks are reset before each strategy's training and each epoch; both peaks include the loaded model and optimizer memory. CPU RSS is sampled at batch boundaries (an estimate). 

Memory numbers across methods are best compared within the same runtime. 

In [ ]:
process = psutil.Process()

def synchronize():
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()

def memory_snapshot():
    result = {"cpu_rss_mb": process.memory_info().rss / 2**20}
    if DEVICE.type == "cuda":
        result["gpu_peak_allocated_mb"] = torch.cuda.max_memory_allocated() / 2**20
        result["gpu_peak_reserved_mb"] = torch.cuda.max_memory_reserved() / 2**20
    else:
        result["gpu_peak_allocated_mb"] = np.nan
        result["gpu_peak_reserved_mb"] = np.nan
    return result

def evaluate(model, loader, collect_predictions=False):
    model.eval()
    predictions, true_values = [], []
    with torch.no_grad()::
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            result = model(input_ids, attention_mask).float().cpu().numpy().reshape(-1)
            predictions.append(result)
            true_values.append(batch["labels"].numpy().reshape(-1))
    y_pred = np.concatenate(predictions)
    y_true = np.concatenate(true_values)
    scores = {
        "rmse": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2": float(r2_score(y_true, y_pred)),
    }
    return (scores, y_true, y_pred) if collect_predictions else scores

def train_one_epoch(model, loader, optimizer, loss_function):
    model.train()
    preds, targets = [], []
    total_squared_error = 0.0
    n_samples = 0
    cpu_peak = process.memory_info().rss / 2**20
    for batch in loader:
        ids = batch["input_ids"].to(DEVICE)
        mask = batch["attention_mask"].to(DEVICE)
        y = batch["labels"].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        predictions = model(ids, mask)
        loss = loss_function(predictions, y)
        loss.backward()
        optimizer.step()
        preds.append(predictions.detach().float().cpu().numpy())
        targets.append(y.detach().float().cpu().numpy())
        total_squared_error += loss.item() * len(y)
        n_samples += len(y)
        cpu_peak = max(cpu_peak, process.memory_info().rss / 2**20)
    train_y = np.concatenate(targets)
    train_p = np.concatenate(preds)
    return {
        "train_mse": total_squared_error / n_samples,
        "train_rmse": float(np.sqrt(mean_squared_error(train_y, train_p))),
        "train_r2": float(r2_score(train_y, train_p)),
    }, cpu_peak

## Run all four methods for the same number of epochs

Keep track of CPU or GPU usage during training

In [ ]:
history_rows = []
final_predictions = {}
summary_rows = []

for method in METHODS:
    print(f"\n{'=' * 72}\n{method} | lr={LEARNING_RATES[method]:g} | epochs={NUM_EPOCHS}")
    train_loader, val_loader = make_loaders(RANDOM_SEED)
    model = build_model(method)
    total_params, trainable_params = count_parameters(model)
    optimizer = torch.optim.AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LEARNING_RATES[method], weight_decay=WEIGHT_DECAY
    )
    criterion = nn.MSELoss()
    synchronize()
    #reset GPU memory peak for each method
    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()
    method_cpu_peak = process.memory_info().rss / 2**20
    start_method = time.perf_counter()
    training_seconds = 0.0
    validation_seconds = 0.0
    method_gpu_allocated_peak = 0.0
    method_gpu_reserved_peak = 0.0
    best_val_rmse = np.inf
    best_epoch = None
    best_predictions = None

    for epoch in range(1, NUM_EPOCHS + 1):
        synchronize()
        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        train_scores, cpu_peak = train_one_epoch(model, train_loader, optimizer, criterion)
        synchronize()
        train_seconds = time.perf_counter() - t0
        training_seconds += train_seconds

        t1 = time.perf_counter()
        val_scores, y_true, y_pred = evaluate(model, val_loader, collect_predictions=True)
        synchronize()
        val_seconds = time.perf_counter() - t1
        validation_seconds += val_seconds

        mem = memory_snapshot()
        method_cpu_peak = max(method_cpu_peak, cpu_peak, mem["cpu_rss_mb"])
        if DEVICE.type == "cuda":
            method_gpu_allocated_peak = max(method_gpu_allocated_peak, mem["gpu_peak_allocated_mb"])
            method_gpu_reserved_peak = max(method_gpu_reserved_peak, mem["gpu_peak_reserved_mb"])

        if val_scores["rmse"] < best_val_rmse:
            best_val_rmse = val_scores["rmse"]
            best_epoch = epoch
            best_predictions = (y_true.copy(), y_pred.copy())

        history_rows.append({
            "method": method, "epoch": epoch, **train_scores,
            "val_rmse": val_scores["rmse"], "val_r2": val_scores["r2"],
            "train_seconds": train_seconds,
            "validation_seconds": val_seconds,
            "epoch_seconds": train_seconds + val_seconds,
            "cpu_rss_peak_sampled_mb": method_cpu_peak,
            "gpu_peak_allocated_mb": mem["gpu_peak_allocated_mb"],
            "gpu_peak_reserved_mb": mem["gpu_peak_reserved_mb"],
        })
        print(f"Epoch {epoch:02d}/{NUM_EPOCHS} | "
              f"Train RMSE {train_scores['train_rmse']:.3f} | "
              f"Val RMSE {val_scores['rmse']:.3f} | "
              f"Val R² {val_scores['r2']:.3f} | "
              f"time {train_seconds + val_seconds:.1f}s")

    synchronize()
    #figure out how much time this took
    elapsed = time.perf_counter() - start_method
    final_predictions[method] = best_predictions
    last = history_rows[-1]
    summary_rows.append({
        "method": method, "total_params": total_params,
        "trainable_params": trainable_params,
        "trainable_pct": 100 * trainable_params / total_params,
        "epochs": NUM_EPOCHS, "best_epoch": best_epoch,
        "best_val_rmse": best_val_rmse,
        "best_val_r2_at_best_rmse": next(
            row["val_r2"] for row in history_rows
            if row["method"] == method and row["epoch"] == best_epoch
        ),
        "final_val_rmse": last["val_rmse"],
        "final_val_r2": last["val_r2"],
        "total_train_seconds": training_seconds,
        "total_validation_seconds": validation_seconds,
        "total_epoch_seconds": training_seconds + validation_seconds,
        "total_wall_seconds": elapsed,
        "peak_gpu_allocated_mb": (method_gpu_allocated_peak if DEVICE.type == "cuda" else np.nan),
        "peak_gpu_reserved_mb": (method_gpu_reserved_peak if DEVICE.type == "cuda" else np.nan),
        "peak_cpu_rss_sampled_mb": method_cpu_peak,
    })
    del model, optimizer, criterion, train_loader, val_loader
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

history_df = pd.DataFrame(history_rows)
summary_df = pd.DataFrame(summary_rows)
display(summary_df.round(3))

##  Plot validation metrics by epoch


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for method, group in history_df.groupby("method", sort=False):
    axes[0].plot(group["epoch"], group["val_rmse"], marker="o", label=method)
    axes[1].plot(group["epoch"], group["val_r2"], marker="o", label=method)
axes[0].set(title="Validation RMSE vs epoch", xlabel="Epoch", ylabel="RMSE (Tm units)")
axes[1].set(title="Validation R² vs epoch", xlabel="Epoch", ylabel="R²")
for ax in axes:
    ax.grid(alpha=0.25)
    ax.set_xticks(range(1, NUM_EPOCHS + 1))
axes[1].legend(loc="best", fontsize=9)
plt.tight_layout()
plt.show()

print("Best validation RMSE for each method (all were trained for equal epochs):")
display(summary_df[["method", "best_epoch", "best_val_rmse", "best_val_r2_at_best_rmse", "final_val_rmse", "final_val_r2"]].round(3))

##  Compare time, memory, and trainable parameters

The time chart totals **training plus validation** across the same number of epochs. The GPU chart shows peak allocated and peak reserved PyTorch CUDA memory. If you run on CPU, GPU memory is unavailable and the notebook instead plots sampled CPU RSS (which includes Python and other live allocations and is only a coarse approximation).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
methods = summary_df["method"].tolist()
x = np.arange(len(methods))
axes[0].bar(x, summary_df["total_train_seconds"], label="Training")
axes[0].bar(x, summary_df["total_validation_seconds"],
            bottom=summary_df["total_train_seconds"], label="Validation")
axes[0].set(ylabel="Seconds", title=f"Time for {NUM_EPOCHS} epochs")
axes[0].legend()

if DEVICE.type == "cuda":
    axes[1].bar(x-0.18, summary_df["peak_gpu_allocated_mb"], width=0.36, label="Allocated")
    axes[1].bar(x+0.18, summary_df["peak_gpu_reserved_mb"], width=0.36, label="Reserved")
    axes[1].set(ylabel="MiB", title="Peak CUDA memory")
    axes[1].legend()
else:
    axes[1].bar(x, summary_df["peak_cpu_rss_sampled_mb"])
    axes[1].set(ylabel="MiB", title="Approx. peak process RSS (CPU)")

axes[2].bar(x, summary_df["trainable_params"] / 1e6)
axes[2].set(ylabel="Millions of parameters", title="Trainable parameters")
for ax in axes:
    ax.set_xticks(x, ["Frozen", "Half", "Full", "LoRA"])
    ax.grid(axis="y", alpha=0.2)
plt.tight_layout()
plt.show()

display(summary_df[[
    "method", "trainable_params", "epochs", "total_train_seconds",
    "total_validation_seconds", "peak_gpu_allocated_mb",
    "peak_gpu_reserved_mb", "peak_cpu_rss_sampled_mb"
]].round(2))

##  Actual versus predicted Tm at each method's best validation epoch


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 10))
for ax, method in zip(axes.flat, METHODS):
    y_true, y_pred = final_predictions[method]
    ax.scatter(y_true, y_pred, alpha=0.35, s=12)
    low, high = min(y_true.min(), y_pred.min()), max(y_true.max(), y_pred.max())
    ax.plot([low, high], [low, high], linestyle="--", color="black")
    ax.set(xlabel="Measured Tm", ylabel="Predicted Tm", title=method)
    ax.grid(alpha=0.15)
plt.tight_layout()
plt.show()
